# Push the trained checkpoint to the Hugging Face Hub

Generated by `scripts/build_kaggle_notebook.py`. No GPU needed (*Accelerator: None*); *Internet* must be on.

**Before running:** (1) *Add-ons -> Secrets*: tick `HF_TOKEN` so this notebook can read it. (2) *Add Input*: add the
output of the training notebook (the version whose run produced the model). Then *Run All*.

In [ ]:
# Finds the checkpoint in the attached training-run output and pushes it to a private Hugging Face Hub repo.
import glob
import os

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

REPO_NAME = "hallucination-detector-modernbert"

api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
repo_id = f"{api.whoami()['name']}/{REPO_NAME}"
ckpts = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/checkpoints/ft/config.json", recursive=True)})
assert len(ckpts) == 1, f"expected one checkpoints/ft in the attached input, found {ckpts or 'none'}"
ckpt = ckpts[0]
print("checkpoint:", ckpt, sorted(os.listdir(ckpt)))
assert any(f.endswith(".safetensors") for f in os.listdir(ckpt)), "no weights file in the checkpoint"

api.create_repo(repo_id, private=True, exist_ok=True)
api.upload_folder(repo_id=repo_id, folder_path=ckpt, commit_message="Run 1: 2 epochs, test F1 0.652, AUROC 0.916")
results = os.path.join(os.path.dirname(os.path.dirname(ckpt)), "results")
for name in ("ft_metrics.json", "ft_train_log.json"):
    if os.path.exists(os.path.join(results, name)):
        api.upload_file(repo_id=repo_id, path_or_fileobj=os.path.join(results, name), path_in_repo=f"results/{name}")
print("uploaded:", sorted(api.list_repo_files(repo_id)))
print(f"https://huggingface.co/{repo_id}")